In [4]:
from langgraph.constants import START, END
from langgraph.graph import StateGraph
from typing import TypedDict


# 输入和输出状态隔离
# 规范输入和输出的状态

# 定义状态
class Mystate(TypedDict):
    query:str
    rag_result:str
    web_result:str
    final_answer:str
    extra_input:str

# 定义输入状态
class InputSchema(TypedDict):
    query:str

# 定义输出状态
class OutputSchema(TypedDict):
    final_answer:str
    extra_input:str

# 构建节点函数
def rag_search_node(state:Mystate):
    query = state['query']
    rag_result = f"这是基于rag的搜索结果: {query}"
    return{
        "rag_result":rag_result
    }
def web_search_node(state:Mystate):
    query = state['query']
    web_result = f"这是基于web的搜索结果: {query}"
    return{
        "web_result":web_result
    }

def final_answer_node(state:Mystate):
    query = state['query']
    rag_result = state['rag_result']
    web_result = state['web_result']
    final_answer = f"这是最终的答案: {query} {rag_result} {web_result}"
    return{
        "final_answer":final_answer
    }

# 创建一个图构造器
builder = StateGraph(state_schema=Mystate,
                    input_schema=InputSchema,
                    output_schema=OutputSchema)

builder.add_node(rag_search_node)
builder.add_node(web_search_node)
builder.add_node(final_answer_node)
builder.add_edge(START,"rag_search_node")
builder.add_edge("rag_search_node", "web_search_node")
builder.add_edge("web_search_node", "final_answer_node")
builder.add_edge("final_answer_node", END)

graph = builder.compile()
result = graph.invoke({"query":"如何使用langgraph?","extra_input":"这是wiwai"})
print(result)





{'final_answer': '这是最终的答案: 如何使用langgraph? 这是基于rag的搜索结果: 如何使用langgraph? 这是基于web的搜索结果: 如何使用langgraph?'}
